# 03 - CNN (deep learning)
A convolutional neural network: small filters slide over the image to detect strokes and curves, so it uses the 2-D shape of the digit.

## 1. Imports

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## 2. Settings
A fixed random seed so results can be repeated, and the folders where the model and results are saved.

In [2]:
SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR, MODELS_DIR, ART_DIR = ROOT / "data", ROOT / "app" / "models", ROOT / "artifacts"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Load the data
Download MNIST (first run only): 60,000 training images and 10,000 test images of handwritten digits.

In [3]:
train = datasets.MNIST(str(DATA_DIR), train=True, download=True)
test = datasets.MNIST(str(DATA_DIR), train=False, download=True)

## 4. Scale the pixels
Pixel values go from 0-255 to 0-1. The digit is white on a black background.

In [4]:
x_all, y_all = train.data.numpy().astype(np.float32) / 255, train.targets.numpy()
x_test, y_test = test.data.numpy().astype(np.float32) / 255, test.targets.numpy()

## 5. Split the data
The 60,000 training images become 54,000 for training and 6,000 for validation (used to compare settings). The 10,000 test images are only used once, at the end.

In [5]:
x_train, x_val, y_train, y_val = train_test_split(x_all, y_all, test_size=6000, stratify=y_all, random_state=SEED)
print(x_train.shape, x_val.shape, x_test.shape)

(54000, 28, 28) (6000, 28, 28) (10000, 28, 28)


## 6. Add a channel
The CNN takes images as (1, 28, 28): one grey channel, then height and width.

In [6]:
x_tr, x_v, x_test_in = (a[:, None] for a in (x_train, x_val, x_test))
print(x_tr.shape)

(54000, 1, 28, 28)


## Device
Use the GPU if there is one, otherwise the CPU.

In [7]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cpu


## 7. The model
Two blocks of Conv -> ReLU -> MaxPool find shapes and shrink the image. Then Flatten -> Dense -> Dropout -> Dense gives the 10 scores.

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self, filters=(16, 32), fc_units=64, dropout=0.3):
        super().__init__()
        f1, f2 = filters
        self.features = nn.Sequential(
            nn.Conv2d(1, f1, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(f1, f2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(f2 * 7 * 7, fc_units), nn.ReLU(), nn.Dropout(dropout), nn.Linear(fc_units, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

## Helper: predict
Runs the model on many images in batches and returns the predicted digit for each.

In [9]:
@torch.no_grad()
def predict(model, x, batch_size=2048):
    model.eval()
    dev = next(model.parameters()).device
    return torch.cat([model(torch.from_numpy(x[i:i + batch_size]).to(dev)).argmax(1).cpu()
                      for i in range(0, len(x), batch_size)]).numpy()

## Helper: train
Adam optimiser with cross-entropy loss. After every epoch it records the loss and the train and validation accuracy. With `restore_best=True` it keeps the weights from the epoch with the best validation accuracy (early stopping).

In [10]:
def train(model, lr, batch_size, epochs, restore_best=False):
    torch.manual_seed(SEED)
    xt, yt = torch.from_numpy(x_tr).to(DEVICE), torch.from_numpy(y_train).long().to(DEVICE)
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_state, start = None, time.perf_counter()
    for epoch in range(epochs):
        model.train()
        idx = torch.randperm(len(xt), device=DEVICE)
        total_loss, correct = 0.0, 0
        for i in range(0, len(xt), batch_size):
            b = idx[i:i + batch_size]
            opt.zero_grad()
            out = model(xt[b])
            loss = loss_fn(out, yt[b])
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(b)
            correct += (out.argmax(1) == yt[b]).sum().item()
        hist["train_loss"].append(total_loss / len(xt))
        hist["train_acc"].append(correct / len(xt))
        hist["val_acc"].append(accuracy_score(y_val, predict(model, x_v)))
        if restore_best and hist["val_acc"][-1] >= max(hist["val_acc"]):
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    hist["train_time_s"] = time.perf_counter() - start
    hist["best_epoch"] = int(np.argmax(hist["val_acc"])) + 1
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist

## Helper: compute test metrics and save the results
Takes the test predictions and writes accuracy, precision, recall, F1, time, size and settings to `artifacts/`, so the comparison notebook can read them.

In [11]:
def save_results(name, model_path, pred_test, train_acc, val_acc, train_time, infer_ms, params, hyper, history=None):
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred_test, average="macro")
    res = dict(model=name, train_acc=train_acc, val_acc=val_acc, test_acc=accuracy_score(y_test, pred_test),
               macro_precision=p, macro_recall=r, macro_f1=f1, test_errors=int((pred_test != y_test).sum()),
               train_time_s=train_time, inference_ms_per_img=infer_ms, parameters=int(params),
               file_size_mb=model_path.stat().st_size / 1e6, hyperparameters=hyper)
    if history is not None:
        res["history"] = history
    (ART_DIR / f"{name.lower()}_results.json").write_text(json.dumps(res, indent=2, default=float))
    return res

## 8. Hyperparameter search
Try 3 filter sizes x 2 learning rates (6 runs, 3 epochs each) and score each on the validation set.

In [12]:
BATCH, FC_UNITS = 128, 64
rows = []
for filters in [(8, 16), (16, 32), (32, 64)]:
    for lr in [1e-3, 3e-3]:
        _, h = train(SimpleCNN(filters, FC_UNITS, 0.0), lr=lr, batch_size=BATCH, epochs=3)
        rows.append(dict(filters=filters, lr=lr, val_acc=h["val_acc"][-1]))

### Pick the best setting
Sort by validation accuracy; the top row wins.

In [13]:
search = pd.DataFrame(rows).sort_values("val_acc", ascending=False).reset_index(drop=True)
best = search.iloc[0]
search

,filters,lr,val_acc
0,"(16, 32)",0.003,0.986167
1,"(8, 16)",0.003,0.984667
2,"(32, 64)",0.003,0.983833
3,"(16, 32)",0.001,0.983500
4,"(32, 64)",0.001,0.983333
5,"(8, 16)",0.001,0.974167


## 9. Train the final model
Retrain with the best settings for more epochs. The epoch with the best validation accuracy is kept.

In [14]:
model, hist = train(SimpleCNN(best["filters"], FC_UNITS, 0.0), lr=best["lr"], batch_size=BATCH, epochs=10, restore_best=True)
print("best epoch:", hist["best_epoch"], "| training time: %.1fs" % hist["train_time_s"])

best epoch: 9 | training time: 84.5s


## 10. Predict on the test set
The first time the model sees the 10,000 test images.

In [15]:
pred_test = predict(model, x_test_in)
print("test accuracy:", accuracy_score(y_test, pred_test))

test accuracy: 0.9895


## 11. Measure the prediction speed
Time for one image, averaged over 1,000 test images.

In [16]:
t0 = time.perf_counter()
predict(model, x_test_in[:1000])
infer_ms = (time.perf_counter() - t0) * 1000 / 1000   # total ms for 1,000 images / 1,000
print("%.4f ms per image" % infer_ms)

0.0530 ms per image


## 12. Save the model
The app needs the weights plus the architecture settings to rebuild the network.

In [17]:
path = MODELS_DIR / "cnn_model.pt"
torch.save({"arch": "cnn", "config": dict(filters=list(best["filters"]), fc_units=FC_UNITS, dropout=0.0), "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, path)
print("saved", path)

saved /Users/seifebeid/Projects/minist/app/models/cnn_model.pt


## 13. Accuracy on the training and validation data
Train accuracy is measured on 10,000 training images; validation accuracy is from the best epoch.

In [18]:
train_acc = accuracy_score(y_train[:10000], predict(model, x_tr[:10000]))
val_acc = hist["val_acc"][hist["best_epoch"] - 1]
print("train acc:", train_acc, "| val acc:", val_acc)

train acc: 0.9969 | val acc: 0.9886666666666667


## 14. Save the results
Writes `artifacts/cnn_results.json` and shows the summary.

In [19]:
res = save_results("CNN", path, pred_test, train_acc=train_acc, val_acc=val_acc, train_time=hist["train_time_s"],
                   infer_ms=infer_ms, params=sum(p.numel() for p in model.parameters()), hyper=dict(filters=list(best["filters"]), lr=best["lr"], fc_units=FC_UNITS, dropout=0.0, batch_size=BATCH), history=hist)
pd.Series({k: v for k, v in res.items() if k not in ("history", "hyperparameters")})

model                         CNN
train_acc                  0.9969
val_acc                  0.988667
test_acc                   0.9895
macro_precision          0.989475
macro_recall             0.989431
macro_f1                 0.989437
test_errors                   105
train_time_s            84.481564
inference_ms_per_img     0.053039
parameters                 105866
file_size_mb             0.427049
dtype: object